# LSTM 做图像分类
前面我们讲了 LSTM 特别适合做序列类型的数据，那么 LSTM 能不能想 CNN 一样用来做图像分类呢？下面我们用 MNIST 手写字体的例子来展示一下如何用 LSTM 做图像分类，但是这种方法并不是主流，这里我们只是作为举例。

对于一张手写字体的图片，其大小是 28 * 28，我们可以将其看做是一个长为 28 的序列，每个序列的特征都是 28，也就是

![MNIST](images/rnn_mnist_classification.jpeg)

这样我们解决了输入序列的问题，对于输出序列怎么办呢？其实非常简单，虽然我们的输出是一个序列，但是我们只需要保留其中一个作为输出结果就可以了，这样的话肯定保留最后一个结果是最好的，因为最后一个结果有前面所有序列的信息。

下面我们直接通过例子展示

In [1]:
import torch
from torch.autograd import Variable
from torch import nn
from torch.utils.data import DataLoader

from torchvision import transforms as tfs
from torchvision.datasets import MNIST

In [2]:
# 定义数据
data_tf = tfs.Compose([
    tfs.ToTensor(),
    tfs.Normalize([0.5], [0.5]) # 标准化
])

train_set = MNIST('../../data/mnist', train=True, transform=data_tf)
test_set  = MNIST('../../data/mnist', train=False, transform=data_tf)

train_data = DataLoader(train_set, 64, True,  num_workers=4)
test_data  = DataLoader(test_set, 128, False, num_workers=4)

In [3]:
# 定义LSTM模型
class LSTM_Classify(nn.Module):
    def __init__(self, in_feature=28, hidden_feature=100, num_class=10, num_layers=2):
        super(LSTM_Classify, self).__init__()
        self.rnn = nn.LSTM(in_feature, hidden_feature, num_layers) # 使用两层 LSTM
        self.classifier = nn.Linear(hidden_feature, num_class) # 将最后一个 rnn 的输出使用全连接得到最后的分类结果
        
    def forward(self, x):
        '''
        x 大小为 (batch, 1, 28, 28)，所以我们需要将其转换成 RNN 的输入形式，即 (28, batch, 28)
        '''
        x = x.squeeze() # 去掉 (batch, 1, 28, 28) 中的 1，变成 (batch, 28, 28)
        x = x.permute(2, 0, 1) # 将最后一维放到第一维，变成 (28, batch, 28)
        out, _ = self.rnn(x) # 使用默认的隐藏状态，得到的 out 是 (28, batch, hidden_feature)
        out = out[-1, :, :] # 取序列中的最后一个，大小是 (batch, hidden_feature)
        out = self.classifier(out) # 得到分类结果
        return out

In [4]:
lstm = LSTM_Classify()
criterion = nn.CrossEntropyLoss()

optimzier = torch.optim.Adam(lstm.parameters(), 1e-2)

In [5]:
# 开始训练
from utils import train
train(lstm, train_data, test_data, 10, optimzier, criterion)

Epoch 0. Train Loss: 0.384897, Train Acc: 0.877432, Valid Loss: 0.128201, Valid Acc: 0.964102, Time 00:00:04
Epoch 1. Train Loss: 0.125357, Train Acc: 0.963686, Valid Loss: 0.087185, Valid Acc: 0.974585, Time 00:00:04
Epoch 2. Train Loss: 0.106303, Train Acc: 0.968417, Valid Loss: 0.103214, Valid Acc: 0.971123, Time 00:00:04
Epoch 3. Train Loss: 0.105601, Train Acc: 0.968967, Valid Loss: 0.108506, Valid Acc: 0.969640, Time 00:00:04
Epoch 4. Train Loss: 0.098099, Train Acc: 0.971065, Valid Loss: 0.091640, Valid Acc: 0.972805, Time 00:00:04
Epoch 5. Train Loss: 0.087544, Train Acc: 0.974031, Valid Loss: 0.098055, Valid Acc: 0.972706, Time 00:00:04
Epoch 6. Train Loss: 0.093329, Train Acc: 0.971665, Valid Loss: 0.093824, Valid Acc: 0.972112, Time 00:00:04
Epoch 7. Train Loss: 0.094267, Train Acc: 0.971915, Valid Loss: 0.100912, Valid Acc: 0.968354, Time 00:00:04
Epoch 8. Train Loss: 0.087802, Train Acc: 0.973481, Valid Loss: 0.089351, Valid Acc: 0.973299, Time 00:00:04
Epoch 9. Train Loss

可以看到，训练 10 次在简单的 mnist 数据集上也取得的了 97% 的准确率，所以说 RNN 也可以做做简单的图像分类，但是这并不是他的主战场，下次课我们会讲到 RNN 的一个使用场景，时间序列预测。

In [7]:
# 定义RNN模型
class RNN_Classify(nn.Module):
    def __init__(self, in_feature=28, hidden_feature=200, num_class=10, num_layers=2):
        super(RNN_Classify, self).__init__()
        self.rnn = nn.RNN(in_feature, hidden_feature, num_layers) # 使用两层 RNN
        self.classifier = nn.Linear(hidden_feature, num_class) # 将最后一个 rnn 的输出使用全连接得到最后的分类结果
        
    def forward(self, x):
        '''
        x 大小为 (batch, 1, 28, 28)，所以我们需要将其转换成 RNN 的输入形式，即 (28, batch, 28)
        '''
        x = x.squeeze() # 去掉 (batch, 1, 28, 28) 中的 1，变成 (batch, 28, 28)
        x = x.permute(2, 0, 1) # 将最后一维放到第一维，变成 (28, batch, 28)
        out, _ = self.rnn(x) # 使用默认的隐藏状态，得到的 out 是 (28, batch, hidden_feature)
        out = out[-1, :, :] # 取序列中的最后一个，大小是 (batch, hidden_feature)
        out = self.classifier(out) # 得到分类结果
        return out

rnn = RNN_Classify()
criterion = nn.CrossEntropyLoss()

optimzier = torch.optim.Adam(rnn.parameters(), 1e-2)

# 开始训练
train(rnn, train_data, test_data, 10, optimzier, criterion)

Epoch 0. Train Loss: 2.447370, Train Acc: 0.101746, Valid Loss: 2.460506, Valid Acc: 0.097013, Time 00:00:03
Epoch 1. Train Loss: 2.425830, Train Acc: 0.099947, Valid Loss: 2.401470, Valid Acc: 0.097013, Time 00:00:03
Epoch 2. Train Loss: 2.430471, Train Acc: 0.100563, Valid Loss: 2.551968, Valid Acc: 0.101266, Time 00:00:03
Epoch 3. Train Loss: 2.431028, Train Acc: 0.101829, Valid Loss: 2.451472, Valid Acc: 0.089597, Time 00:00:03
Epoch 4. Train Loss: 2.423556, Train Acc: 0.101979, Valid Loss: 2.362008, Valid Acc: 0.113627, Time 00:00:03
Epoch 5. Train Loss: 2.433166, Train Acc: 0.099813, Valid Loss: 2.496187, Valid Acc: 0.097607, Time 00:00:03
Epoch 6. Train Loss: 2.438123, Train Acc: 0.102845, Valid Loss: 2.438180, Valid Acc: 0.100475, Time 00:00:03
Epoch 7. Train Loss: 2.442176, Train Acc: 0.099847, Valid Loss: 2.430335, Valid Acc: 0.096123, Time 00:00:03
Epoch 8. Train Loss: 2.437061, Train Acc: 0.098664, Valid Loss: 2.448695, Valid Acc: 0.100475, Time 00:00:03
Epoch 9. Train Loss

In [8]:
# 定义GRU模型
class GRU_Classify(nn.Module):
    def __init__(self, in_feature=28, hidden_feature=200, num_class=10, num_layers=2):
        super(GRU_Classify, self).__init__()
        self.rnn = nn.GRU(in_feature, hidden_feature, num_layers) # 使用两层 GRU
        self.classifier = nn.Linear(hidden_feature, num_class) # 将最后一个 GRU 的输出使用全连接得到最后的分类结果
        
    def forward(self, x):
        '''
        x 大小为 (batch, 1, 28, 28)，所以我们需要将其转换成 RNN 的输入形式，即 (28, batch, 28)
        '''
        x = x.squeeze() # 去掉 (batch, 1, 28, 28) 中的 1，变成 (batch, 28, 28)
        x = x.permute(2, 0, 1) # 将最后一维放到第一维，变成 (28, batch, 28)
        out, _ = self.rnn(x) # 使用默认的隐藏状态，得到的 out 是 (28, batch, hidden_feature)
        out = out[-1, :, :] # 取序列中的最后一个，大小是 (batch, hidden_feature)
        out = self.classifier(out) # 得到分类结果
        return out

gru = GRU_Classify()
criterion = nn.CrossEntropyLoss()

optimzier = torch.optim.Adam(gru.parameters(), 1e-2)

# 开始训练
train(gru, train_data, test_data, 10, optimzier, criterion)

Epoch 0. Train Loss: 0.283351, Train Acc: 0.913346, Valid Loss: 0.370029, Valid Acc: 0.887856, Time 00:00:06
Epoch 1. Train Loss: 0.488758, Train Acc: 0.848148, Valid Loss: 0.302515, Valid Acc: 0.903877, Time 00:00:08
Epoch 2. Train Loss: 0.324206, Train Acc: 0.900553, Valid Loss: 0.279081, Valid Acc: 0.916634, Time 00:00:07
Epoch 3. Train Loss: 0.305605, Train Acc: 0.907832, Valid Loss: 0.374469, Valid Acc: 0.881922, Time 00:00:07
Epoch 4. Train Loss: 0.280624, Train Acc: 0.915678, Valid Loss: 0.294950, Valid Acc: 0.914062, Time 00:00:07
Epoch 5. Train Loss: 0.296740, Train Acc: 0.910698, Valid Loss: 0.322287, Valid Acc: 0.904964, Time 00:00:08
Epoch 6. Train Loss: 0.288669, Train Acc: 0.913563, Valid Loss: 0.242535, Valid Acc: 0.930479, Time 00:00:07
Epoch 7. Train Loss: 0.277322, Train Acc: 0.918543, Valid Loss: 0.223355, Valid Acc: 0.931072, Time 00:00:08
Epoch 8. Train Loss: 0.285646, Train Acc: 0.912797, Valid Loss: 0.274894, Valid Acc: 0.917326, Time 00:00:09
Epoch 9. Train Loss